# Module 3.4: Cost-Sensitive Evaluation, PR-AUC & Optuna Hyperparameter Tuning
### 🇵🇱 Ewaluacja Kosztowa, Krzywe PR i Optymalizacja Hiperparametrów z Optuna

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/max480425/industrial-ml-course/blob/main/public/notebooks/m03_l04_crossval_roc_cost_matrix_optuna.ipynb)

**Industrial Machine Learning & Predictive Maintenance Interactive Laboratory**

> **Summary:** Implement TimeSeriesSplit cross-validation, align models with real financial failure costs, and run Optuna Bayesian hyperparameter optimization.  
> 🇵🇱 **Streszczenie:** Zaimplementuj walidację krzyżową TimeSeriesSplit, powiąż model z finansowym kosztem awarii i zoptymalizuj hiperparametry z Optuna.

- **Estimated Time:** 30 minutes
- **Runtime:** Python 3.10+ (CPU or Free Tier T4 GPU)
- **Packages:** `numpy`, `scipy`, `scikit-learn`, `pandas`, `matplotlib`, `seaborn`
---


In [ ]:
# Step 1: Install & Verify Python Scientific Environment
!pip install -q numpy scipy scikit-learn pandas matplotlib seaborn

import numpy as np
import pandas as pd
import scipy.signal as signal
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.family'] = 'monospace'
print("✓ Industrial ML environment loaded successfully.")


### Key Engineering Concepts:
1. **TimeSeriesSplit**: Standard k-fold cross-validation shuffles data across time, leaking future sensor trends into past predictions. Forward walk-forward splits (`TimeSeriesSplit`) preserve physical causality.
2. **Asymmetric Business Penalties**:
   - $\text{Cost(False Negative)} = \$10,000$ (catastrophic production line stoppage).
   - $\text{Cost(False Positive)} = \$50$ (5-minute technician inspection).
3. **Threshold Economics**: The academic default threshold of $\tau = 0.50$ is economically disastrous. Tuning $\tau^*$ via Optuna minimizes total financial loss.


In [ ]:
# 1. Simulate Plant Defect Probabilities & Financial Cost Function
np.random.seed(42)
n_batches = 3000
defect_rate = 0.025  # 2.5% failure rate

y_true = (np.random.rand(n_batches) < defect_rate).astype(int)

# Simulated predicted failure probability
pred_probs = np.zeros(n_batches)
pred_probs[y_true == 0] = np.random.beta(a=1.2, b=10.0, size=(y_true == 0).sum())
pred_probs[y_true == 1] = np.random.beta(a=5.0, b=1.8, size=(y_true == 1).sum())

COST_FN = 10000.0  # $10,000 per missed failure
COST_FP = 50.0     # $50 per false alarm

def calculate_plant_cost(y_actual, probabilities, threshold):
    preds = (probabilities >= threshold).astype(int)
    fn = ((y_actual == 1) & (preds == 0)).sum()
    fp = ((y_actual == 0) & (preds == 1)).sum()
    return fn * COST_FN + fp * COST_FP


In [ ]:
# 2. Sweep Thresholds and Compute Real Dollar Losses
thresholds = np.linspace(0.01, 0.90, 150)
dollar_losses = [calculate_plant_cost(y_true, pred_probs, t) for t in thresholds]

best_idx = np.argmin(dollar_losses)
optimal_thresh = thresholds[best_idx]
min_loss = dollar_losses[best_idx]
loss_default_050 = calculate_plant_cost(y_true, pred_probs, 0.50)

plt.figure(figsize=(11, 5))
plt.plot(thresholds, dollar_losses, color='#ef4444', linewidth=2.5, label='Total Financial Loss ($)')
plt.axvline(optimal_thresh, color='#10b981', linestyle='--', linewidth=2, 
            label=f'Cost-Optimal Threshold = {optimal_thresh:.2f} (Loss: ${min_loss:,.0f})')
plt.axvline(0.50, color='#64748b', linestyle=':', label=f'Academic 0.50 Threshold (Loss: ${loss_default_050:,.0f})')

plt.title("Asymmetric Cost Optimization: Minimizing Dollar Loss vs Decision Threshold", fontsize=13, fontweight='bold')
plt.xlabel("Decision Threshold (tau)")
plt.ylabel("Total Expected Loss [USD]")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Optimal Threshold saves ${loss_default_050 - min_loss:,.0f} compared to the default 0.50 threshold!")


## 📋 Practical Hands-On Checklist

- [ ] **Task 1:** Implement TimeSeriesSplit cross-validation to prevent future-data contamination *(🇵🇱 Zaimplementuj walidację krzyżową TimeSeriesSplit, zapobiegając wyciekom z przyszłości)*
- [ ] **Task 2:** Calculate Precision-Recall Area Under Curve (PR-AUC) under 99.5% class imbalance *(🇵🇱 Wylicz pole pod krzywą PR (PR-AUC) przy niezbalansowaniu klas na poziomie 99.5%)*
- [ ] **Task 3:** Run an Optuna Bayesian study minimizing plant financial loss function ($10,000 FN vs $50 FP) *(🇵🇱 Przeprowadź badanie bayesowskie w Optuna minimalizujące funkcję strat ($10 000 za FN vs $50 za FP))*


In [ ]:
# Exercise Challenge:
# Assert that optimal threshold saves at least $20,000 over default 0.50 threshold
savings = loss_default_050 - min_loss
print(f"Verified Net Dollar Savings: ${savings:,.0f}")
assert savings >= 20000, "Cost optimization failed to achieve minimum financial threshold."
print("✓ Financial cost matrix optimization verified.")
